# 08 — Tactical constraints and corrected neural search

This notebook repairs the observed failure to inspect an immediate capture reply. It keeps the AHR Scout/Worker network, exact Go rules and all legal moves. The fixed Queen explicitly schedules high-risk move/reply witnesses, and PUCT uses parent-value first-play urgency, a modest score component and transparent local tactical biases.

**Inputs:** tagged definitions from 00–03; the preserved final development-game record and checkpoint from the first AHR v2 run. No opponent-generated training labels. **Outputs:** revised search definitions, family-separated policy-only tactical fixtures, validation report and fresh-kernel recovery fixture. **Expected runtime:** 2–5 minutes on CPU; no long training or persistent model changes. **Checkpoints:** disposable search recovery fixtures in `checkpoints/ahr-v2-repair-search-*`; production model/replay checkpoints belong to notebook 09.

**Claims:** capture witnesses are exact only within the stated one- or two-ply scope. Tactical biases and neural search are estimates. UNKNOWN on time/node cutoff remains UNKNOWN. Legal self-atari is neither banned nor treated as a proved game loss.

**Resume:** notebook 09 saves `go2_search_state_dict(tree)` with model, all RNGs and active-game state; restore through `go2_search_from_state_dict` using the same frozen configuration and source identities. Run All here validates a fresh-process restore before notebook 10 launches the long run.

## Load tagged definitions

In [ ]:
from pathlib import Path
import nbformat, json, os, sys, copy, time
PROJECT=Path.cwd()
while not (PROJECT/'AGENTS.md').exists() and PROJECT.parent!=PROJECT: PROJECT=PROJECT.parent
if not (PROJECT/'AGENTS.md').exists(): raise RuntimeError('Open this notebook within the AHR Go workspace.')
base=nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4)
for cell in base.cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(compile(cell.source,'00 ahr-export','exec'),globals())
for filename in ['01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb']:
    go2_import_exports(PROJECT/GO2_ROOT/filename,globals())
torch.set_num_threads(2)
print('Imported tagged definitions only. Existing notebooks and checkpoints remain unchanged.')

## Exact local witnesses and family-separated labels

In [ ]:
GO2_REPAIR_SEARCH_NOTEBOOK='08_tactical_constraints_and_search.ipynb'

def go2_tactical_summary(state):
    """Exact immediate captures and maximum opponent one-move capture witnesses.

    This is a finite local calculation, not a full-game win/loss assertion.
    Every legal action remains present, including legal non-suicide self-atari.
    """
    if state.terminal: return {}
    side=state.to_play
    own_before=sum(v==side for v in state.board)
    enemy_before=sum(v==-side for v in state.board)
    result={}
    for action in go2_legal_actions(state):
        child=go2_step(state,int(action))
        captured=enemy_before-sum(v==-side for v in child.board)
        losses={}
        if not child.terminal:
            liberties=set()
            for stones,libs in go2_groups(child.board,child.size):
                if child.board[next(iter(stones))]==side and len(libs)==1: liberties.update(libs)
            for reply in sorted(liberties):
                grandchild=go2_try_step(child,int(reply))
                if grandchild is not None:
                    removed=sum(v==side for v in child.board)-sum(v==side for v in grandchild.board)
                    if removed: losses[int(reply)]=int(removed)
        risk=max(losses.values(),default=0)
        utility=math.tanh((captured-risk)/3.0)
        result[int(action)]={'captured':int(captured),'risk':int(risk),'reply_losses':losses,
            'utility':float(utility),'verified_plies':2,'status':'EXACT_BOUNDED',
            'scope':'immediate captures and maximum single legal next-reply capture'}
    return result


def go2_tactical_policy_targets(state):
    summary=go2_tactical_summary(state)
    if not summary: return None
    own_threats=[len(stones) for stones,libs in go2_groups(state.board,state.size)
                 if state.board[next(iter(stones))]==state.to_play and len(libs)==1]
    max_capture=max(v['captured'] for v in summary.values())
    if own_threats:
        best=min(v['risk']-v['captured'] for v in summary.values())
        actions=[a for a,v in summary.items() if v['risk']-v['captured']==best]
        kind='minimize_immediate_capture_loss_minus_captures'
    elif max_capture:
        actions=[a for a,v in summary.items() if v['captured']==max_capture]
        kind='maximize_immediate_captured_stones'
    else: return None
    policy=np.zeros(state.pass_action+1,dtype=np.float64);policy[actions]=1/len(actions)
    return {'policy_target':policy.tolist(),'kind':kind,'nonterminal_outcome':None,
        'verification':{'status':'EXACT_BOUNDED','depth':2,'all_legal_candidates':len(summary),
        'goal':kind,'full_game_optimality':False,'history_digest':go2_state_digest(state)}}


def go2_tactical_family(state):
    # Side-relative D4 canonical family is assigned BEFORE any augmentation.
    variants=[tuple(v*state.to_play for v in go2_transform_state(state,k).board) for k in range(8)]
    return go2_digest(['capture_defence_setup_family_v1',state.size,min(variants)])


def go2_tactical_positions(seed=2026100708,count=64,split='train',size=9):
    if split not in ('train','dev','lockbox'): raise ValueError('Invalid tactic split.')
    if count<1 or size<5: raise ValueError('Tactical fixtures require count>0 and size>=5.')
    rng=np.random.default_rng(seed); rows=[]; seen=set();attempts=0
    while len(rows)<count and attempts<count*300:
        attempts+=1; board=[0]*(size*size);length=int(rng.integers(1,min(4,size-2)))
        row=int(rng.integers(1,size-1));column=int(rng.integers(1,size-length))
        color=int(rng.choice([-1,1]));stones={row*size+column+i for i in range(length)}
        surround=set().union(*(set(go2_neighbors(size)[s]) for s in stones))-stones
        liberty=(row+1)*size+column+int(rng.integers(length))
        for s in stones: board[s]=color
        for s in surround-{liberty}: board[s]=-color
        protected=stones|surround|set().union(*(set(go2_neighbors(size)[s]) for s in surround))
        # Independent distant context varies genuine setup families; all groups
        # must have a liberty, and D4/color equivalents remain the same family.
        free=[s for s in range(size*size) if s not in protected]
        rng.shuffle(free)
        for s in free[:int(rng.integers(0,min(12,len(free))+1))]: board[s]=int(rng.choice([-1,1]))
        if any(not libs for _,libs in go2_groups(board,size)): continue
        side=color if rng.random()<.5 else -color
        state=go2_state_from_board(board,to_play=side,size=size,komi=7.5)
        family=go2_tactical_family(state); bucket=int(family[:8],16)%10
        assigned='dev' if bucket==8 else 'lockbox' if bucket==9 else 'train'
        if assigned!=split or family in seen: continue
        target=go2_tactical_policy_targets(state)
        if target is None: continue
        seen.add(family);rows.append({'state':state,'family':family,'split':assigned,**target})
    if len(rows)!=count: raise RuntimeError('Unable to fill disjoint tactical family bank.')
    return rows


def go2_bounded_capture_proof(state,target_points,attacker,depth=2,node_budget=256):
    """All-legal minimax proof of capturing original stones within bounded plies.
    UNKNOWN means cutoff/depth exhaustion; it never becomes UNSAT or proved loss.
    A REFUTED_WITHIN_BOUND result requires complete enumeration, including pass.
    """
    target_points=tuple(sorted(set(int(x) for x in target_points)))
    if not target_points or attacker not in (-1,1) or depth<0 or node_budget<1: raise ValueError('Invalid local proof request.')
    defender=-attacker
    if any(state.board[p]!=defender for p in target_points): raise ValueError('Targets must initially contain defender stones.')
    visited=0
    def solve(position,remaining):
        nonlocal visited
        if all(position.board[p]!=defender for p in target_points): return 'PROVED_CAPTURE_WITHIN_BOUND'
        if remaining==0 or position.terminal: return 'REFUTED_WITHIN_BOUND'
        if visited>=node_budget:return 'UNKNOWN'
        visited+=1; outcomes=[]
        for action in go2_legal_actions(position):
            outcome=solve(go2_step(position,action),remaining-1);outcomes.append(outcome)
            if position.to_play==attacker and outcome=='PROVED_CAPTURE_WITHIN_BOUND':return outcome
            if position.to_play==defender and outcome=='REFUTED_WITHIN_BOUND':return outcome
        if 'UNKNOWN' in outcomes:return 'UNKNOWN'
        return ('REFUTED_WITHIN_BOUND' if position.to_play==attacker else 'PROVED_CAPTURE_WITHIN_BOUND')
    status=solve(state,depth)
    return {'status':status,'depth':depth,'nodes':visited,'node_budget':node_budget,
        'goal':'capture specified original defender stones within depth','all_legal_replies':True,
        'full_game_optimality':False}


## Search configuration, identities and expansion

In [ ]:
import dataclasses, copy, math, time
import numpy as np

@dataclasses.dataclass(frozen=True)
class Go2SearchConfig:
    simulations: int = 128
    batch_size: int = 8
    c_puct: float = 1.5
    virtual_loss: float = 1.0
    prior_floor: float = 1e-6
    seed: int = 2026100603
    max_depth: int = 256
    wall_time_seconds: float | None = None
    root_noise_fraction: float = 0.0
    root_dirichlet_alpha: float = 0.3

    fpu_reduction: float = 0.25
    score_value_weight: float = 0.15
    tactical_prior_weight: float = 0.30
    tactical_edge_weight: float = 0.60
    tactical_reply_reservations: int = 16

    def __post_init__(self):
        if not 0<=self.score_value_weight<=1 or not 0<=self.tactical_prior_weight<=1 or not 0<=self.tactical_edge_weight<=1 or self.fpu_reduction<0 or self.tactical_reply_reservations<0:
            raise ValueError('Invalid repaired search settings.')
        if self.simulations < 1 or self.batch_size < 1 or self.max_depth < 1:
            raise ValueError('Simulation, batch and depth budgets must be positive.')
        if self.c_puct <= 0 or self.virtual_loss < 0 or self.prior_floor <= 0:
            raise ValueError('Invalid PUCT, reservation or exploration settings.')
        if self.wall_time_seconds is not None and self.wall_time_seconds < 0:
            raise ValueError('Wall-clock budget cannot be negative.')
        if not 0 <= self.root_noise_fraction <= 1 or self.root_dirichlet_alpha <= 0:
            raise ValueError('Invalid root exploration noise settings.')


def _go2_search_game():
    return {
        'legal': go2_legal_actions,
        'step': go2_step,
        'key': go2_cache_key,
        'dump': go2_state_to_dict,
        'load': go2_state_from_dict,
        'terminal': lambda state: bool(state.terminal),
        'value': lambda state: float(go2_outcome(state)['value_current']),
        'actions': lambda state: int(state.size * state.size + 1),
    }


def _go2_search_identity(model, state, config, model_identity, context, project, game):
    if model_identity is None:
        if model is None:
            raise ValueError('A custom evaluator requires an explicit immutable model identity.')
        model_identity = go2_model_identity(model)
    return {
        'schema': 'ahr-go-v2-search-tree-v2-tactical',
        'model': str(model_identity),
        'sources': {**go2_source_identity(go2_project(project)),GO2_ROOT+'/'+GO2_REPAIR_SEARCH_NOTEBOOK:go2_code_sha(go2_project(project)/GO2_ROOT/GO2_REPAIR_SEARCH_NOTEBOOK)},
        'config': dataclasses.asdict(config),
        'root': go2_digest(game['key'](state)),
        'context': go2_digest({} if context is None else context),
    }


def _go2_node(state, game):
    terminal = game['terminal'](state)
    return {'state': state, 'terminal': terminal, 'expanded': terminal,
            'value': game['value'](state) if terminal else None,
            'visits': 0, 'virtual_visits': 0, 'edges': {}}


def _go2_new_tree(identity, state, config, game):
    rng = np.random.default_rng(config.seed)
    return {'identity': identity, 'config': dataclasses.asdict(config),
            'root': 0, 'next_node': 1, 'nodes': {0: _go2_node(state, game)},
            'tactical_queue': [], 'tactical_reserved_replies': [],
            'cache': {}, 'pending': [], 'rng': rng,
            'completed_simulations': 0, 'elapsed_seconds': 0.0,
            'counters': {'neural_batches': 0, 'neural_positions': 0,
                         'cache_hits': 0, 'duplicate_reservations': 0,
                         'depth_cutoffs': 0, 'terminal_simulations': 0,
                         'atomic_batches': 0},
            'queen': {'kind': 'fixed_batch_schedule', 'batch_size': config.batch_size}}


def _go2_inference_key(tree, state, game):
    return go2_digest({'state_and_rules': game['key'](state),
                       'model': tree['identity']['model'],
                       'context': tree['identity']['context']})


def _go2_expand(tree, node_id, output, config, game):
    node = tree['nodes'][node_id]
    if node['terminal']:
        raise AssertionError('A terminal position cannot require neural inference.')
    prior = np.asarray(output['prior'], dtype=np.float64)
    value = float(output['value'])
    action_count = game['actions'](node['state'])
    legal = sorted(int(action) for action in game['legal'](node['state']))
    if prior.shape != (action_count,) or not np.isfinite(prior).all() or np.any(prior < 0):
        raise ValueError('Scout output must contain finite nonnegative action probabilities.')
    if not math.isfinite(value) or not -1.000001 <= value <= 1.000001:
        raise ValueError('Worker value must be finite and in [-1, 1].')
    if not legal:
        raise AssertionError('A nonterminal position must have legal actions.')
    illegal = np.ones(action_count, dtype=bool)
    illegal[legal] = False
    if np.any(prior[illegal] > 1e-8):
        raise ValueError('Scout probabilities assign mass to an illegal action.')
    masked = np.zeros(action_count, dtype=np.float64)
    masked[legal] = np.maximum(prior[legal], config.prior_floor)
    masked /= masked.sum()
    if node_id == tree['root'] and config.root_noise_fraction:
        noise = tree['rng'].dirichlet(np.full(len(legal), config.root_dirichlet_alpha))
        masked[legal] = ((1-config.root_noise_fraction)*masked[legal]
                         + config.root_noise_fraction*noise)
    node['value'] = max(-1.0, min(1.0, value))
    tactics=go2_tactical_summary(node['state']) if node_id==tree['root'] and isinstance(node['state'],Go2State) else {}
    if tactics:
        heuristic=np.zeros(action_count,dtype=np.float64)
        heuristic[legal]=np.exp([2*tactics[a]['utility'] for a in legal]);heuristic/=heuristic.sum()
        masked=(1-config.tactical_prior_weight)*masked+config.tactical_prior_weight*heuristic
        priorities=sorted(((v['risk'],v['captured'],a,r) for a,v in tactics.items() for r in v['reply_losses']),reverse=True)
        tree['tactical_queue']=[{'action':a,'reply':r} for _,_,a,r in priorities[:config.tactical_reply_reservations]]
    node['tactical']=tactics
    node['edges'] = {action: {'prior': float(masked[action]), 'n': 0, 'w': 0.0,
                            'virtual_n': 0, 'child': None, 'tactical_bias': config.tactical_edge_weight*tactics[action]['utility'] if tactics else 0.0} for action in legal}
    node['expanded'] = True

## Worker plus bounded score utility

In [ ]:
def _go2_evaluate_nodes(tree, node_ids, evaluator, config, game):
    missing = {}
    for node_id in node_ids:
        node = tree['nodes'][node_id]
        if node['terminal'] or node['expanded']:
            continue
        key = _go2_inference_key(tree, node['state'], game)
        if key in tree['cache']:
            tree['counters']['cache_hits'] += 1
            _go2_expand(tree, node_id, tree['cache'][key], config, game)
        else:
            missing.setdefault(key, []).append(node_id)
    if not missing:
        return
    keys = list(missing)
    states = [tree['nodes'][missing[key][0]]['state'] for key in keys]
    outputs = evaluator(states)
    if len(outputs) != len(states):
        raise ValueError('Evaluator result count differs from the inference batch.')
    validated = []
    for key, output in zip(keys, outputs):
        if 'network_version' in output and str(output['network_version']) != tree['identity']['model']:
            raise ValueError('Evaluator model identity changed during an active search.')
        minimal = {'prior': np.asarray(output['prior'], dtype=np.float64).tolist(),
                   'value': float((1-config.score_value_weight)*float(output['value'])+config.score_value_weight*math.tanh(float(output.get('score',0.0))*3.0))}
        # Validate every output on a disposable tree before committing any expansion.
        probe = copy.deepcopy(tree['nodes'][missing[key][0]])
        check_tree = {'nodes': {1: probe}, 'root': 0, 'rng': tree['rng']}
        _go2_expand(check_tree, 1, minimal, config, game)
        validated.append((key, minimal))
    tree['counters']['neural_batches'] += 1
    tree['counters']['neural_positions'] += len(states)
    for key, minimal in validated:
        tree['cache'][key] = minimal
        for node_id in missing[key]:
            _go2_expand(tree, node_id, minimal, config, game)

## Queen reservations, first-play urgency and real visits

In [ ]:
def _go2_select_path(tree, config, game, reserved=()):
    reserved = set(reserved)
    # The fixed Queen reserves exact tactical witness paths before ordinary PUCT.
    # A child is evaluated first; its specified legal capture reply is forced on
    # the next available batch. Queue and reservations are checkpointed.
    root=tree['nodes'][tree['root']]
    for item in list(tree.get('tactical_queue',[])):
        action,reply=item['action'],item['reply'];edge=root['edges'][action]
        if edge['child'] is None:
            child_id=tree['next_node'];tree['next_node']+=1
            tree['nodes'][child_id]=_go2_node(game['step'](root['state'],action),game);edge['child']=child_id
        child_id=edge['child'];child=tree['nodes'][child_id]
        if not child['expanded']:
            if child_id not in reserved:return [(tree['root'],action)],child_id,False
            continue
        if child['terminal'] or reply not in child['edges']:
            tree['tactical_queue'].remove(item);continue
        reply_edge=child['edges'][reply]
        if reply_edge['n']:
            tree['tactical_queue'].remove(item);continue
        if reply_edge['child'] is None:
            leaf_id=tree['next_node'];tree['next_node']+=1
            tree['nodes'][leaf_id]=_go2_node(game['step'](child['state'],reply),game);reply_edge['child']=leaf_id
        leaf_id=reply_edge['child']
        if leaf_id not in reserved:
            tree['tactical_reserved_replies'].append([action,reply])
            return [(tree['root'],action),(child_id,reply)],leaf_id,False
    def descend(node_id, path):
        node = tree['nodes'][node_id]
        depth_cutoff = len(path) >= config.max_depth
        if node['terminal'] or not node['expanded'] or depth_cutoff:
            if node_id in reserved and not node['terminal']:
                tree['counters']['duplicate_reservations'] += 1
                return None
            return path, node_id, bool(depth_cutoff and not node['terminal'])
        total = node['visits'] + node['virtual_visits']
        scored = []
        for action, edge in node['edges'].items():
            count = edge['n'] + edge['virtual_n']
            q = ((edge['w'] - config.virtual_loss * edge['virtual_n']) / count
                 if count else float(node['value'])-config.fpu_reduction)
            exploration = config.c_puct * edge['prior'] * math.sqrt(1+total) / (1+count)
            scored.append((q + exploration + edge.get('tactical_bias',0.0), edge['prior'], -action, action))
        # Reservations exclude a leaf only for this inference batch. They never
        # remove a legal edge from the persistent Scout action set.
        for _, _, _, action in sorted(scored, reverse=True):
            edge = node['edges'][action]
            if edge['child'] is None:
                child = game['step'](node['state'], action)
                child_id = tree['next_node']
                tree['next_node'] += 1
                tree['nodes'][child_id] = _go2_node(child, game)
                edge['child'] = child_id
            found = descend(edge['child'], [*path, (node_id, action)])
            if found is not None:
                return found
        return None
    return descend(tree['root'], [])


def _go2_reserve(tree, path, leaf_id):
    for node_id, action in path:
        tree['nodes'][node_id]['virtual_visits'] += 1
        tree['nodes'][node_id]['edges'][action]['virtual_n'] += 1
    tree['nodes'][leaf_id]['virtual_visits'] += 1


def _go2_release(tree, path, leaf_id):
    for node_id, action in path:
        tree['nodes'][node_id]['virtual_visits'] -= 1
        tree['nodes'][node_id]['edges'][action]['virtual_n'] -= 1
    tree['nodes'][leaf_id]['virtual_visits'] -= 1


def _go2_collect_batch(tree, config, game, count):
    if tree['pending']:
        raise AssertionError('Resolve a pending batch before collecting another.')
    reserved = set()
    while len(tree['pending']) < count:
        selected = _go2_select_path(tree, config, game, reserved)
        if selected is None:
            break
        path, leaf_id, depth_cutoff = selected
        if not tree['nodes'][leaf_id]['terminal']:
            reserved.add(leaf_id)
        _go2_reserve(tree, path, leaf_id)
        tree['pending'].append({'path': path, 'leaf': leaf_id,
                                'depth_cutoff': bool(depth_cutoff),
                                'simulation_id': tree['completed_simulations'] + len(tree['pending'])})
    if not tree['pending']:
        raise AssertionError('A nonterminal root must permit at least one search simulation.')


def _go2_resolve_batch(tree, evaluator, config, game):
    pending = tree['pending']
    if not pending:
        return
    _go2_evaluate_nodes(tree, [record['leaf'] for record in pending], evaluator, config, game)
    # Once evaluation succeeds, this entire batch is an atomic real-visit commit.
    for record in pending:
        path, leaf_id = record['path'], record['leaf']
        leaf = tree['nodes'][leaf_id]
        value = float(leaf['value'])
        _go2_release(tree, path, leaf_id)
        leaf['visits'] += 1
        for node_id, action in reversed(path):
            value = -value
            node = tree['nodes'][node_id]
            node['edges'][action]['n'] += 1
            node['edges'][action]['w'] += value
            node['visits'] += 1
        tree['completed_simulations'] += 1
        tree['counters']['depth_cutoffs'] += int(record['depth_cutoff'])
        tree['counters']['terminal_simulations'] += int(leaf['terminal'])
    tree['pending'] = []
    tree['counters']['atomic_batches'] += 1
    go2_validate_search_tree(tree)

## Conservation and corruption checks

In [ ]:
def go2_validate_search_tree(tree):
    nodes = tree['nodes']
    if tree['root'] not in nodes:
        raise ValueError('Search root is absent.')
    simulation_ids = [record['simulation_id'] for record in tree['pending']]
    if len(simulation_ids) != len(set(simulation_ids)):
        raise ValueError('A pending batch double-counts a simulation identity.')
    pending_leaf_ids = [record['leaf'] for record in tree['pending'] if not nodes[record['leaf']]['terminal']]
    if len(pending_leaf_ids) != len(set(pending_leaf_ids)):
        raise ValueError('A pending batch contains duplicate leaf reservations.')
    expected_node_virtual = {node_id: 0 for node_id in nodes}
    expected_edge_virtual = {(node_id, action): 0 for node_id, node in nodes.items()
                             for action in node['edges']}
    for record in tree['pending']:
        node_id = tree['root']
        for parent_id, action in record['path']:
            if parent_id != node_id or action not in nodes[parent_id]['edges']:
                raise ValueError('A pending path is inconsistent with the tree.')
            edge = nodes[parent_id]['edges'][action]
            node_id = edge['child']
            expected_node_virtual[parent_id] += 1
            expected_edge_virtual[(parent_id, action)] += 1
        if node_id != record['leaf'] or node_id not in nodes:
            raise ValueError('A pending leaf is inconsistent with its path.')
        expected_node_virtual[node_id] += 1
    for node_id, node in nodes.items():
        if node['visits'] < 0 or node['virtual_visits'] != expected_node_virtual[node_id]:
            raise ValueError('Search node visit conservation failed.')
        if node['expanded'] and not node['terminal'] and not node['edges']:
            raise ValueError('Expanded nonterminal node has no legal edges.')
        for action, edge in node['edges'].items():
            if edge['n'] < 0 or edge['virtual_n'] != expected_edge_virtual[(node_id, action)]:
                raise ValueError('Search edge reservation conservation failed.')
            if not math.isfinite(edge['w']) or abs(edge['w']) > edge['n'] + 1e-6:
                raise ValueError('Search accumulated values are invalid.')
            if edge['child'] is not None:
                if edge['child'] not in nodes or nodes[edge['child']]['visits'] != edge['n']:
                    raise ValueError('Child visits and parent edge visits differ.')
        if sum(edge['n'] for edge in node['edges'].values()) > node['visits']:
            raise ValueError('Outgoing visits exceed total node visits.')
    root = nodes[tree['root']]
    if root['visits'] != tree['completed_simulations']:
        raise ValueError('Completed simulations and root visits differ.')
    if root['expanded'] and not root['terminal']:
        if sum(edge['n'] for edge in root['edges'].values()) != tree['completed_simulations']:
            raise ValueError('Root policy includes incomplete or duplicate simulations.')
    return True

## Search and full state serialization

In [ ]:
def go2_search_state_dict(tree, _game=None):
    game = _go2_search_game() if _game is None else _game
    go2_validate_search_tree(tree)
    payload = {key: copy.deepcopy(value) for key, value in tree.items() if key not in ('nodes', 'rng')}
    payload['rng'] = copy.deepcopy(tree['rng'].bit_generator.state)
    payload['nodes'] = {}
    for node_id, node in tree['nodes'].items():
        saved = {key: copy.deepcopy(value) for key, value in node.items() if key != 'state'}
        saved['state'] = game['dump'](node['state'])
        payload['nodes'][int(node_id)] = saved
    payload['payload_sha256'] = go2_digest(payload)
    return payload


def go2_search_from_state_dict(payload, model, state, config=None, model_identity=None,
                               context=None, project=None, _game=None):
    game = _go2_search_game() if _game is None else _game
    if _game is None:
        go2_verify_replay(state)
    payload = copy.deepcopy(payload)
    digest = payload.pop('payload_sha256', None)
    if digest is None or digest != go2_digest(payload):
        raise ValueError('Search payload checksum is invalid; refusing to restart.')
    config = Go2SearchConfig(**payload['config']) if config is None else config
    identity = _go2_search_identity(model, state, config, model_identity, context, project, game)
    if payload['identity'] != identity:
        raise ValueError('Search model, source, configuration, rules/history or context changed.')
    rng = np.random.default_rng()
    rng.bit_generator.state = payload.pop('rng')
    nodes = {}
    for node_id, saved in payload.pop('nodes').items():
        saved['state'] = game['load'](saved['state'])
        saved['edges'] = {int(action): edge for action, edge in saved['edges'].items()}
        nodes[int(node_id)] = saved
    tree = {**payload, 'nodes': nodes, 'rng': rng}
    tree['cache'] = {str(key): value for key, value in tree['cache'].items()}
    go2_validate_search_tree(tree)
    return tree


def go2_search(model, state, config=None, tree=None, stop_after_simulations=None,
               on_batch=None, evaluator=None, model_identity=None, context=None,
               project=None, _game=None):
    config = Go2SearchConfig() if config is None else config
    game = _go2_search_game() if _game is None else _game
    identity = _go2_search_identity(model, state, config, model_identity, context, project, game)
    if tree is None:
        if _game is None:
            go2_verify_replay(state)
        tree = _go2_new_tree(identity, state, config, game)
    elif tree['identity'] != identity:
        raise ValueError('Active search identity differs from this request.')
    go2_validate_search_tree(tree)
    root = tree['nodes'][tree['root']]
    if root['terminal']:
        return {'action': None, 'policy_target': [0.0]*game['actions'](state),
                'visit_counts': [0]*game['actions'](state), 'tree': tree,
                'status': 'TERMINAL', 'search_status': 'EXACT_TERMINAL',
                'move_legality': 'NO_MOVE_TERMINAL', 'value': root['value'],
                'completed_simulations': 0, 'counters': copy.deepcopy(tree['counters'])}
    if evaluator is None:
        if model is None:
            raise ValueError('Provide a model or an explicit evaluator.')
        evaluator = lambda states: go2_eval_batch(model, states)
    started = time.monotonic()
    stop = config.simulations if stop_after_simulations is None else min(config.simulations, int(stop_after_simulations))
    if stop < tree['completed_simulations']:
        raise ValueError('Resume stop position precedes completed simulations.')
    budget_cutoff = False
    try:
        if tree['pending']:
            _go2_resolve_batch(tree, evaluator, config, game)
            tree['elapsed_seconds'] += time.monotonic() - started
            started = time.monotonic()
            if on_batch is not None:
                on_batch(tree)
        if not root['expanded']:
            if config.wall_time_seconds == 0:
                budget_cutoff = True
            else:
                _go2_evaluate_nodes(tree, [tree['root']], evaluator, config, game)
        while root['expanded'] and tree['completed_simulations'] < stop:
            elapsed = tree['elapsed_seconds'] + time.monotonic() - started
            if config.wall_time_seconds is not None and elapsed >= config.wall_time_seconds:
                budget_cutoff = True
                break
            # A pause threshold never changes the configured atomic batch shape.
            # It may stop just after the requested simulation count.
            remaining = config.simulations-tree['completed_simulations']
            _go2_collect_batch(tree, config, game, min(config.batch_size, remaining))
            _go2_resolve_batch(tree, evaluator, config, game)
            tree['elapsed_seconds'] += time.monotonic() - started
            started = time.monotonic()
            if on_batch is not None:
                on_batch(tree)
    finally:
        tree['elapsed_seconds'] += time.monotonic()-started
    counts = np.zeros(game['actions'](state), dtype=np.int64)
    for action, edge in root['edges'].items():
        counts[action] = edge['n']
    policy = counts.astype(np.float64)
    if policy.sum():
        policy /= policy.sum()
        action = max(root['edges'], key=lambda a: (root['edges'][a]['n'], root['edges'][a]['w']/root['edges'][a]['n']+root['edges'][a].get('tactical_bias',0.0) if root['edges'][a]['n'] else -float('inf'), root['edges'][a]['prior'], -a))
    else:
        legal = sorted(game['legal'](state))
        action = max(root['edges'], key=lambda a: (root['edges'][a]['prior'], -a)) if root['edges'] else legal[0]
        # No completed simulation means no training policy target.
    status = ('BUDGET_CUTOFF' if budget_cutoff else
              'COMPLETE' if tree['completed_simulations'] >= config.simulations else 'PAUSED')
    root_q = {action: edge['w']/edge['n'] if edge['n'] else None
              for action, edge in root['edges'].items()}
    return {'action': int(action), 'policy_target': policy.tolist(),
            'visit_counts': counts.tolist(), 'tree': tree, 'status': status,
            'search_status': 'UNKNOWN', 'move_legality': 'LEGAL',
            'policy_target_valid': bool(counts.sum()),
            'value': None if root['value'] is None else float(root['value']), 'root_q': root_q,
            'completed_simulations': tree['completed_simulations'],
            'elapsed_seconds': tree['elapsed_seconds'],
            'counters': copy.deepcopy(tree['counters']),
            'queen': copy.deepcopy(tree['queen']), 'tactical_verification':copy.deepcopy(root.get('tactical',{})), 'tactical_reserved_replies':copy.deepcopy(tree.get('tactical_reserved_replies',[]))}

In [ ]:
Go2RepairSearchConfig=Go2SearchConfig
go2_repair_search=go2_search
go2_repair_search_state_dict=go2_search_state_dict
go2_repair_search_from_state_dict=go2_search_from_state_dict


## Bounded regressions and fresh-kernel recovery

In [ ]:
def go2_repair_search_validation(project=None):
    import uuid, subprocess, sys
    project=go2_project(project);checks={};started=time.monotonic()
    inherited=go2_search_validation(project)
    checks.update({'core_'+k:v for k,v in inherited['checks'].items()})
    # Local capture/defence labels are independently replayed; illegal mass is zero.
    train=go2_tactical_positions(count=24,split='train');dev=go2_tactical_positions(count=12,split='dev')
    checks['family_split_before_D4']=not(set(x['family'] for x in train)&set(x['family'] for x in dev))
    checks['tactical_policy_legal_and_nonterminal_outcome_absent']=all(
        x['nonterminal_outcome'] is None and sum(x['policy_target'])>0.999999 and
        all(not p or a in go2_legal_actions(x['state']) for a,p in enumerate(x['policy_target'])) for x in train+dev)
    checks['D4_family_stable']=all(go2_tactical_family(go2_transform_state(x['state'],k))==x['family'] for x in train[:4] for k in range(8))
    snap=go2_state_from_board((-1,-1,1,0,-1,0,-1,1,1),size=3,to_play=1,komi=0)
    s1=go2_step(snap,3);s2=go2_step(s1,5);s3=go2_step(s2,2)
    snap_summary=go2_tactical_summary(snap)
    checks['legal_snapback_self_atari_not_banned']=3 in go2_legal_actions(snap) and snap_summary[3]['captured']==1 and snap_summary[3]['risk']==1 and sum(v==-1 for v in s2.board)-sum(v==-1 for v in s3.board)==4
    proof=go2_bounded_capture_proof(snap,(6,),1,depth=1,node_budget=100)
    checks['one_ply_capture_exact_bounded_proof']=proof['status']=='PROVED_CAPTURE_WITHIN_BOUND'
    safe=go2_state_from_board([0,0,0,0,-1,0,0,0,0],size=3,to_play=1,komi=0)
    cutoff=go2_bounded_capture_proof(safe,(4,),1,depth=3,node_budget=1)
    checks['node_budget_cutoff_UNKNOWN']=cutoff['status']=='UNKNOWN'
    # Reproduce the observed failure from the preserved development game only.
    old_report_path=project/'reports/ahr_v2/evaluations/ahr-v2-selfplay-20261006-12h-final-dev-6080/report.json'
    old_report=json.loads(old_report_path.read_text()); old_game=old_report['games'][3]
    failed_state=go2_replay(go2_initial_state(),old_game['state']['actions'][:51])
    checkpoint_path=project/'checkpoints/ahr-v2-selfplay-20261006-12h'/old_report['checkpoint']['file']
    if go2_file_sha(checkpoint_path)!=old_report['checkpoint']['sha256']:raise ValueError('Preserved checkpoint hash mismatch.')
    old_checkpoint=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
    model=Go2Network(Go2NetConfig(board_size=9,**old_checkpoint['config']['network']))
    model.load_state_dict(old_checkpoint['model']);model.eval()
    config=Go2SearchConfig(simulations=128,batch_size=8)
    before=time.monotonic();search=go2_search(model,failed_state,config,project=project)
    elapsed=time.monotonic()-before
    root=search['tree']['nodes'][0];bad=root['edges'][59];child=search['tree']['nodes'][bad['child']]
    checks['observed_F3_C1_capture_reply_gets_real_visit']=child['edges'][74]['n']>=1
    checks['observed_F3_immediate_eleven_stone_loss_detected']=root['tactical'][59]['reply_losses'][74]==11
    checks['observed_search_selects_lower_immediate_risk_than_F3']=root['tactical'][search['action']]['risk']<root['tactical'][59]['risk']
    checks['all_legal_actions_still_eligible']=set(root['edges'])==set(go2_legal_actions(failed_state))
    checks['observed_policy_real_visit_sum']=sum(search['visit_counts'])==128
    # Save an active reservation batch after 8 committed simulations. Restore
    # with a fresh Jupyter process and compare every scientific tree field.
    cfg=Go2SearchConfig(simulations=64,batch_size=8,seed=202610070809)
    uninterrupted=go2_search(model,failed_state,cfg,project=project)['tree']
    partial=go2_search(model,failed_state,cfg,stop_after_simulations=8,project=project)['tree']
    _go2_collect_batch(partial,cfg,_go2_search_game(),8)
    checks['unfinished_tactical_queue_is_checkpointed']=bool(partial['tactical_queue'])
    search_payload=go2_search_state_dict(partial)
    token=uuid.uuid4().hex[:10];run_dir=project/'checkpoints'/('ahr-v2-repair-search-'+token)
    identity={'schema':GO2_SCHEMA,'sources':{**go2_source_identity(project),GO2_ROOT+'/'+GO2_REPAIR_SEARCH_NOTEBOOK:go2_code_sha(project/GO2_ROOT/GO2_REPAIR_SEARCH_NOTEBOOK)}}
    payload={'schema':GO2_SCHEMA,'identity':identity,'model':model.state_dict(),'optimizer':{},'scheduler':{},'scaler':{},
        'rng':go2_capture_rng(),'sampler':{},'global_step':8,'config':{'search':dataclasses.asdict(cfg),'network':dataclasses.asdict(model.config)},
        'metrics':{'completed_simulations':8},'active_search':search_payload,'root_state':go2_state_to_dict(failed_state)}
    with go2_checkpoint_lock(run_dir):entry=go2_save_checkpoint(run_dir,payload,resume_cell='08 validation: active search restore',best=True)
    child_path=project/'reports/ahr_v2/fixtures'/('repair-search-resume-'+token+'.ipynb');child_path.parent.mkdir(parents=True,exist_ok=True)
    child_source="""from pathlib import Path
import nbformat, torch, dataclasses, copy
project=Path(PROJECT_LITERAL)
for name in ['00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','08_tactical_constraints_and_search.ipynb']:
 for cell in nbformat.read(project/'experiments/ahr_v2'/name,4).cells:
  if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,name,'exec'),globals())
torch.set_num_threads(2)
saved,entry=go2_load_checkpoint(project/'checkpoints'/RUN_LITERAL)
model=Go2Network(Go2NetConfig(**saved['config']['network']));model.load_state_dict(saved['model']);model.eval()
state=go2_state_from_dict(saved['root_state']);cfg=Go2SearchConfig(**saved['config']['search'])
tree=go2_search_from_state_dict(saved['active_search'],model,state,cfg,project=project)
assert len(tree['pending'])>0 and tree['completed_simulations']==8 and tree['tactical_queue']
out=go2_search(model,state,cfg,tree=tree,project=project)
payload=go2_search_state_dict(out['tree']);payload.pop('payload_sha256');payload.pop('elapsed_seconds')
go2_atomic_json(project/'reports/ahr_v2/fixtures'/RESULT_LITERAL,{'digest':go2_digest(payload),'completed':out['completed_simulations']})
print('Fresh kernel restored pending reservations and reached',out['completed_simulations'])
""".replace('PROJECT_LITERAL',repr(str(project))).replace('RUN_LITERAL',repr(run_dir.name)).replace('RESULT_LITERAL',repr('repair-search-resume-'+token+'.json'))
    child_nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(child_source)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(child_nb,child_path)
    result=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=180',str(child_path)],cwd=project,capture_output=True,text=True,timeout=240)
    if result.returncode:raise RuntimeError(result.stdout+'\n'+result.stderr)
    fresh=json.loads(child_path.with_suffix('.json').read_text())
    baseline=go2_search_state_dict(uninterrupted);baseline.pop('payload_sha256');baseline.pop('elapsed_seconds')
    checks['fresh_kernel_active_tactical_search_exact_resume']=fresh['completed']==64 and fresh['digest']==go2_digest(baseline)
    corrupted=copy.deepcopy(search_payload);corrupted['completed_simulations']+=1
    try:go2_search_from_state_dict(corrupted,model,failed_state,cfg,project=project);corrupt_rejected=False
    except ValueError:corrupt_rejected=True
    checks['corrupt_active_search_refuses_restart']=corrupt_rejected
    report={'schema':'ahr-v2-tactical-search-validation-v1','status':'PASS' if all(checks.values()) else 'FAIL','checks':checks,
        'sources':identity['sources'],'old_checkpoint_sha256':old_report['checkpoint']['sha256'],
        'counterfactual':{'development_game':3,'before_ply':52,'bad_action':59,'capture_reply':74,'capture_reply_real_visits':child['edges'][74]['n'],
        'selected_action':search['action'],'selected_immediate_risk':root['tactical'][search['action']]['risk'],
        'F3_immediate_risk':root['tactical'][59]['risk'],'C1_other_group_risk':root['tactical'][74]['risk'],
        'simulations':128,'elapsed_seconds':elapsed},'tactical_train_families':len(train),'tactical_dev_families':len(dev),
        'resume_fixture':str(child_path.relative_to(project)),'checkpoint':str(run_dir.relative_to(project)),
        'runtime_seconds':time.monotonic()-started,'claim':'Bounded tactical witnesses and recovery validated; no new win rate established.'}
    go2_atomic_json(project/'reports/ahr_v2/tactical_search_validation.json',report)
    if not all(checks.values()):raise AssertionError(report)
    return report


In [ ]:
validation=go2_repair_search_validation(PROJECT)
from IPython.display import display, Markdown
import pandas as pd
display(Markdown('**Validation: '+validation['status']+'** — '+str(len(validation['checks']))+' checks. No new trained weights.'))
with pd.option_context('display.max_colwidth',None):
    display(pd.DataFrame([{'Check':k.replace('_',' '),'Result':'PASS' if v else 'FAIL'} for k,v in validation['checks'].items()]))
observed=validation['counterfactual']
def coordinate(action):
    return 'PASS' if action==81 else 'ABCDEFGHJ'[action%9]+str(9-action//9)
display(pd.DataFrame([
    {'Measure':'Failed move / capture reply','Observed':coordinate(observed['bad_action'])+' / '+coordinate(observed['capture_reply'])},
    {'Measure':'Verified capture reply visits','Observed':str(observed['capture_reply_real_visits'])},
    {'Measure':'Selected move','Observed':coordinate(observed['selected_action'])},
    {'Measure':'Maximum next-reply loss: F3 / selected move','Observed':str(observed['F3_immediate_risk'])+' / '+str(observed['selected_immediate_risk'])+' stones'},
    {'Measure':'Search budget','Observed':str(observed['simulations'])+' real visits'},
    {'Measure':'CPU search time','Observed':format(observed['elapsed_seconds'],'.3f')+' seconds'}
]))

## Interpretation and next notebook

The preserved model now receives an explicit, real search visit for the failed F3 → C1 capture response. The table compares the same checkpoint and development position; it does **not** measure improved match strength. C1 saves the eleven-stone group but leaves a different threatened group, so its other-group risk must not be described as zero.

Notebook 09 adds family-separated tactical policy supervision and lowers outcome overfitting with fresh complete-game replay, regularization, softer labels and balanced updates. Notebook 10 validates recovery and launches the detached 8-hour experiment; notebook 11 provides the human board. The final lockbox remains untouched.